## Modified Stochastic Subspace Descent for Nonlinear Least Squares

### Gauss-Newton

Oracle: ${r_i(x)}^m_{i=1} = {q^\top \nabla r_i(x)}^m_{i=1} = J(x)q$ where q is a directional derivative. 

We know $r(x_k + p) = r(x_k) + J(x_k)p$ is the residual function. The nonlinear least square problem is then $$ \min_x ||r(x_k) + J(x_k)p||^2_2 $$

Apply $p = Pn$ to the SSD algorithm, where P is a projection matrix of size $d \times l$ where $l < d$. After applying the substitution, we obtain the normal equation 
\begin{equation}
-(JP)^\top r = (JP)^\top (JP) n
\end{equation}
solving for $n^*$ we have 
$$ Pn = p = P(P^\top J^\top J P)^{-1} P^\top J^\top r $$
Let $P = Q \cdot S$, where the column space of P and Q is the same and $S^{-1}$ exists. Then 
$$ p = Q(Q^\top J^\top J Q)^{-1} Q^\top J^\top r $$
Let $\tilde{J} = J \cdot Q$, then 
$$ p = Q ( \tilde{J}^\top \tilde{J} ) ^{-1} \tilde{J}^\top r$$
and the algorithm costs $l$ for $l$ directional derivatives. 

For computing purposes, compute by finding
$$ p = \text{arg} \min_{n \in \mathbb{R}} ||\tilde{J}n - r_x||^2_2$$

In [11]:
import jax
import jax.numpy as jnp
import numpy as np
from scipy.optimize import line_search

import os
os.environ["PATH"] = "/opt/homebrew/bin:" + os.environ["PATH"]  # Apple Silicon
# os.environ["PATH"] = "/usr/local/bin:" + os.environ["PATH"]   # Intel Mac
import pycutest

RuntimeError: Could not find MASTSIF folder - has MASTSIF environment variable been set correctly?

In [10]:
def gn_ssd(x, l, r, alpha=1, max_iter=200):
    '''
    input: x is the starting point; P is dxl; r the residual function
    Q has Haar distribution
    Assume alpha = 1
    iteration: x = P 
    '''
    d = x.size
    rng = np.random.default_rng(seed)
    history = []
    
    for n in range(max_iter):
        rx = r(x)
        history.append(0.5 * float(rx @ rx))
        Z = np.random.default_rng().standard_normal((dim, l))
        Q, R = linalg.qr(Z, mode='economic') #economic mode for finding the QR so it's not a full square matrix

        #use jax.jvp for jacobian-vector multiplication. Need x and v in tuple. Returns r(x) and Jx
        #use jax.vmap to map it to every row
        J_tild = jax.vmap(lambda v: jax.jvp(r, (x,), (v,))[1])(Q.T).T
        p = jnp.linalg.lstsq(J_tild.T, rx)[0]
        x = x - Q @ p

### Levenberg-Marquardt
L-M is a combination between Gauss-Newton and gradient descent. The problem is (after evaluating Lagrange Multiplier):
$$ \min_p ||Jp + r||^2 + \frac{\lambda}{2}||x_0|| $$
The normal equation for L-M is 
$$ -(JP)^\top r = ((JP)^\top (JP)-\lambda I) n $$
Input: 
* initial $\lambda_0$
* initial $x_0$ size $n \times 1$
* r

Importantly, need to update $\lambda$ each step. I use $D = I$ because we don't compute the whole Jacobian. 

Figure out how to update $\lambda$. Derive the update equation for this. 

7/2/2026 Did not finish this, found a Cartis paper that already does random newton in trust region. No need to follow up. 

In [8]:
def lm_ssd(x, l, r, lam, max_iter=200):
    '''
    for ...:
        evaluate x
        evaluate rho, the actual reduction
        if rho < :
        else: 
        
    '''

    for n in range(max_iter):
        rx = r(x)
        
        Z = np.random.default_rng().standard_normal((dim, l))
        Q, R = linalg.qr(Z, mode='economic') #economic mode for finding the QR so it's not a full square matrix

        #use jax.jvp for jacobian-vector multiplication. Need x and v in tuple. Returns r(x) and Jx
        #use jax.vmap to map it to every row
        J_tild = jax.vmap(lambda v: jax.jvp(r, (x,), (v,))[1])(Q.T)
        p = jnp.linalg.lstsq(J_tild.T @ J_tild - lam @ np.eye(l), - J_tild.T @ rx)[0]

        #compute gain ratio

## Linesearch For SSD
7/6/2026  
Let $$\phi(t) = f(x_k + t p_k) $$ where $\phi(t): \mathbb{R} \rightarrow \mathbb{R} $  
Let $\Phi(t) \approx \phi(t)$, where $\Phi(t)$ is a quadratic/cubic interpolation of points $\phi(0)$, $\phi(t_1)$, and $\phi'(0)$ (and $\phi(t_2)$ if cubic).   
For SSD, $\phi'(0) = - || p_k||^2 $, so I will modify the scipy code where there's no need to calculate derphi. 

In [ ]:
#input alpha and l, independent variables
#0 < alpha < 2l/(d*lambda). Why?

def ssd(x, l, f, dim, alpha=None, max_iter=200):
    '''
    followed algorithm 2.1:
    1: Inputs: α, l. step size, subspace rank
    2: Initialize: x0 . arbitrary initialization
    3: for k = 1, 2, . . . do
    4: Generate P_k
    5: Apply recursion
    '''

    fx = []
    x_history = [np.array(x)]
    gradientFcn = jax.jit(jax.grad(f))
    for n in range(max_iter):
        #generate P using haar distribution (not necessary)
        Z = np.random.default_rng().standard_normal((dim, l)) # 
        Q, R = linalg.qr(Z, mode='economic') #economic mode for finding the QR so it's not a full square matrix
        #lambda step is optional, only required for Haar. 
        # Lambda = np.diag(np.sign(np.diag(R))) # R_ii > 0, ensure unique QR fact. 
        P = np.sqrt(dim / l) * Q 

        #recurse
        grad = gradientFcn(x)
        y = P.T @ grad
        step = P @ y
        if alpha is None:
            p_k = - step
            result = line_search(
                    lambda x: float(f(jnp.array(x))),
                    lambda x: np.array(gradientFcn(jnp.array(x))),
                    np.array(x),
                    p_k,
                )
            alpha = result[0]
            x = x + alpha * p_k
        else:
            x = x - alpha* step
        
        #plug x into function
        fx.append(float(f(jnp.array(x))))
        x_history.append(np.array(x))

    return fx, x_history